# 08 — Darkstore-Serviceability Fusion

Adds a **confidence-aware quick-commerce serviceability layer** to the master feature store, fusing in the
geocoded Blinkit / Zepto / Swiggy Instamart darkstores (`web/darkstores.json`, a **partial sample**).

**Non-negotiables:** *absence ≠ unserviceable* (`Unknown` means unknown, not unreachable); `city_coverage_confidence`
is a **proxy** (no ground-truth darkstore totals); centroid-precision `Confirmed` is softer than locality-precision.

Pure logic lives in `nb08lib.py` (unit-tested). This notebook wires the real Nominatim geocoder + I/O.
Reads `artifacts/localities_features_master.parquet` + `web/darkstores.json`; writes
`artifacts/localities_master_serviceable.parquet` + CSV + xlsx.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import nb08lib as L

ART = Path.cwd() / "artifacts"

def find_file(name):
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(name)

df = pd.read_parquet(ART / "localities_features_master.parquet").reset_index(drop=True)
darkstores = L.load_darkstores(str(find_file("web/darkstores.json")))
print("localities:", len(df), "| darkstores:", len(darkstores))

localities: 1001 | darkstores: 4081


In [2]:
# Coordinate refinement (real Nominatim, cached, rate-limited, offline fallback)
import time
from geopy.geocoders import Nominatim

CACHE = ART / "coord_cache.json"
cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
geolocator = Nominatim(user_agent="goatlife-nb08", timeout=10)

def geocode_fn(query):
    if query in cache:
        v = cache[query]
        return tuple(v) if v else None
    try:
        time.sleep(1.1)
        loc = geolocator.geocode(query)
        res = [loc.latitude, loc.longitude] if loc else None
    except Exception:
        res = None
    cache[query] = res
    CACHE.write_text(json.dumps(cache))
    return tuple(res) if res else None

city_centroids = (df.dropna(subset=["lat", "lng"]).groupby("ADDRESS")[["lat", "lng"]].mean()
                  .apply(tuple, axis=1).to_dict())
records = df[["AREA", "ADDRESS", "lat", "lng"]].to_dict("records")
stats = L.refine_coordinates(records, geocode_fn, city_centroids)
df["lat_r"] = [r["lat_r"] for r in records]
df["lng_r"] = [r["lng_r"] for r in records]
df["coord_precision"] = [r["coord_precision"] for r in records]
print("refinement:", stats)
print(df["coord_precision"].value_counts(dropna=False).to_dict())

refinement: {'refined': 596, 'pincode': 168, 'no_geo': 115}
{'locality': 718, 'pincode': 168, 'no-geo': 115}


In [3]:
# Per-brand distances + confirmation
grid = L.build_grid(darkstores)
rows = []
for r in df.itertuples():
    lat_r, lng_r = r.lat_r, r.lng_r
    if lat_r is None or (isinstance(lat_r, float) and np.isnan(lat_r)):
        rows.append({"nearest_blinkit_km": np.nan, "nearest_zepto_km": np.nan, "nearest_swiggy_km": np.nan,
                     "nearest_known_darkstore_km": np.nan, "n_darkstores_within_3km": 0,
                     "blinkit_confirmed": False, "zepto_confirmed": False, "swiggy_confirmed": False,
                     "n_brands_confirmed": 0, "brands_confirmed_list": ""})
        continue
    res = L.nearest_by_brand(lat_r, lng_r, grid)
    confirm_km, _ = L.precision_radii(r.coord_precision)
    conf = L.confirmed_brands(res["per_brand"], confirm_km)
    rows.append({
        "nearest_blinkit_km": res["per_brand"]["Blinkit"],
        "nearest_zepto_km": res["per_brand"]["Zepto"],
        "nearest_swiggy_km": res["per_brand"]["Swiggy Instamart"],
        "nearest_known_darkstore_km": res["nearest_any"],
        "n_darkstores_within_3km": res["n_within_3km"],
        "blinkit_confirmed": "Blinkit" in conf,
        "zepto_confirmed": "Zepto" in conf,
        "swiggy_confirmed": "Swiggy Instamart" in conf,
        "n_brands_confirmed": len(conf),
        "brands_confirmed_list": "+".join(conf),
    })
df = pd.concat([df, pd.DataFrame(rows, index=df.index)], axis=1)
print("confirmed (>=1 brand):", int((df.n_brands_confirmed > 0).sum()), "/", len(df))

confirmed (>=1 brand): 792 / 1001


In [4]:
# City coverage confidence (tertiles) + per-brand presence assertion (guards normalization bug)
from collections import Counter
target_cities = set(df["ADDRESS"])
city_counts = Counter(d["city"] for d in darkstores if d["city"] in target_cities)
buckets = L.coverage_confidence_buckets(dict(city_counts))
df["city_known_darkstores"] = df["ADDRESS"].map(dict(city_counts)).fillna(0).astype(int)
df["city_coverage_confidence"] = df["ADDRESS"].map(buckets)

present = {(d["city"], d["brand"]) for d in darkstores}
for city in df["ADDRESS"].unique():
    for b in L.BRANDS:
        assert (city, b) in present, f"NORMALIZATION BUG: {city} missing {b}"
print("per-brand presence OK for all cities")
print("city coverage:", buckets)

per-brand presence OK for all cities
city coverage: {'Chennai': 'Medium', 'Hyderabad': 'High', 'Mumbai': 'High', 'Bangalore': 'High', 'New Delhi': 'High', 'Lucknow': 'Low', 'Pune': 'Medium', 'Gurugram': 'Low', 'Kolkata': 'Medium', 'Chandigarh': 'Low'}


In [5]:
# Serviceability state, confidence, gtm action
city_median = df.groupby("ADDRESS")["dist_to_city_centroid_km"].median().to_dict()

def row_state(r):
    return L.assign_state(r["nearest_known_darkstore_km"], r["coord_precision"],
                          r.get("dist_to_city_centroid_km"), city_median.get(r["ADDRESS"]),
                          r["city_coverage_confidence"])

df["serviceability_state"] = df.apply(row_state, axis=1)
df["serviceability_confidence"] = df.apply(
    lambda r: L.serviceability_confidence(r["serviceability_state"], r["coord_precision"], r["city_coverage_confidence"]),
    axis=1)
df["gtm_action"] = df.apply(lambda r: L.gtm_action(r["icp_verdict"], r["serviceability_state"]), axis=1)

In [6]:
# Validation
print("serviceability_state:", df["serviceability_state"].value_counts(dropna=False).to_dict())
print("gtm_action:", df["gtm_action"].value_counts().to_dict())
conf_pct = df.groupby("ADDRESS").apply(lambda x: round((x.serviceability_state == "Confirmed").mean() * 100)).sort_values(ascending=False)
print("\nConfirmed % by city:")
print(conf_pct.to_string())
print("\nSpot-checks:")
for area in ["Koramangala", "Sushant Lok", "Banjara Hills", "Attibele"]:
    m = df[df.AREA.str.contains(area, case=False, na=False)]
    if len(m):
        r = m.iloc[0]
        print(f"  {r.AREA[:30]:30} {r.serviceability_state:10} nearest={r.nearest_known_darkstore_km} "
              f"brands={r.brands_confirmed_list or '-'} -> {r.gtm_action}")
# coordinate-collapse fix worked? Bangalore should now show varied states/distances
bg = df[df.ADDRESS == "Bangalore"]
print(f"\nBangalore distinct states: {bg.serviceability_state.nunique()} | "
      f"distinct nearest_km: {bg.nearest_known_darkstore_km.round(1).nunique()}")

serviceability_state: {'Confirmed': 792, 'Unknown': 190, 'Likely': 19}
gtm_action: {'SAMPLE + QC test': 450, 'HOLD': 378, 'PUSH-NOW': 97, 'SAMPLE (D2C / offline)': 59, 'D2C / OFFLINE - verify QC': 17}

Confirmed % by city:
ADDRESS
New Delhi     99
Bangalore     91
Pune          90
Mumbai        85
Kolkata       83
Chennai       83
Hyderabad     77
Gurugram      74
Lucknow       74
Chandigarh    35

Spot-checks:
  Block 1st Koramangala, Bangalo Confirmed  nearest=0.41 brands=Blinkit+Zepto+Swiggy Instamart -> SAMPLE + QC test
  Block A Sushant Lok Phase 1, G Confirmed  nearest=2.84 brands=Blinkit+Zepto+Swiggy Instamart -> SAMPLE + QC test
  Banjara Hills, Hyderabad       Confirmed  nearest=0.62 brands=Blinkit+Zepto+Swiggy Instamart -> PUSH-NOW
  Attibele, Bangalore            Unknown    nearest=7.69 brands=- -> HOLD

Bangalore distinct states: 3 | distinct nearest_km: 21


In [7]:
# Save
out = ART / "localities_master_serviceable.parquet"
df.to_parquet(out, index=False)
df.to_csv(Path.cwd() / "localities_master_serviceable.csv", index=False)
df.to_excel(Path.cwd() / "localities_master_serviceable.xlsx", index=False)
print("Saved", df.shape, "->", out.name)

Saved (1001, 146) -> localities_master_serviceable.parquet


## Output
`artifacts/localities_master_serviceable.parquet` + CSV + xlsx — the master store enriched with per-brand
darkstore proximity, a precision-aware 3-state serviceability (`Confirmed`/`Likely`/`Unknown`),
`serviceability_confidence`, and a combined `gtm_action`.

**Honesty:** `Unknown` ≠ unserviceable (a darkstore may simply be missing from the sample);
`city_coverage_confidence` is a proxy; centroid-precision confirmations are softer than locality-precision ones.